### Примеры работы модели

Чтобы запустить код, необходимо использовать **Run All**, далее появятся два виджета, в которых можно настроить параметры запуска и построить анимации.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact_manual
from IPython.display import display, clear_output
import time

np.random.seed(42)

In [2]:
from src.structs import Generator, Model, State
from src.utils import grid4, grid8, cluster_graph

После этой ячейки появится виджет для настройки параметров запуска модели на сетке

In [3]:
GRID_BUILDERS = {
    "grid4": grid4,
    "grid8": grid8,
}

out = widgets.Output()
status = widgets.HTML(value="<b>Готово</b>")

def run_simulation(_):
    status.value = "<b style='color:orange'>Идёт расчёт...</b>"

    steps = steps_w.value
    n = n_w.value
    m = m_w.value
    beta = beta_w.value
    T = T_w.value
    grid_kind = grid_w.value

    with out:
        clear_output(wait=True)
        print("🚀 Запуск модели...")
        print("⚙️ Симуляция...")

    gen = Generator(mean=0.5, std=0.1)
    nodes = GRID_BUILDERS[grid_kind](gen, n, m)

    center = (n // 2) * m + (m // 2)
    nodes[center].state = State.SPREADER

    model = Model(nodes, beta=beta, T=T)
    model.run(steps=steps, record=True)

    with out:
        clear_output(wait=True)
        display(model.plot(steps, "grid", n=n, m=m, interval=10))

    status.value = "<b style='color:green'>Готово</b>"

style = {'description_width': '150px'}
steps_w = widgets.IntSlider(value=500, min=10, max=2000, step=10, description="Время моделирования", style=style)
n_w = widgets.IntSlider(value=100, min=10, max=300, step=10, description="n", style=style)
m_w = widgets.IntSlider(value=100, min=10, max=300, step=10, description="m", style=style)

grid_w = widgets.Dropdown(options=["окрестность Неймана", "окрестность Мура"], value="окрестность Неймана", description="Выбор сетки", style=style)

beta_w = widgets.FloatSlider(value=0.8, min=0.0, max=1.0, step=0.01, description="Интересность", style=style)
T_w = widgets.IntSlider(value=24, min=1, max=200, step=1, description="Время актуальности", style=style)

steps_box = widgets.HBox([
    steps_w,
    widgets.Label("ч.", layout=widgets.Layout(width="20px"))
])

T_box = widgets.HBox([
    T_w,
    widgets.Label("ч.", layout=widgets.Layout(width="20px"))
])

button = widgets.Button(
    description="Run simulation",
    button_style="success",
    icon="play"
)

button.on_click(run_simulation)

ui = widgets.VBox([
    widgets.HBox([steps_box, grid_w]),
    widgets.HBox([n_w, m_w]),
    widgets.HBox([beta_w, T_box]),
    widgets.HBox([button, status])
])

display(ui, out)

Output()

После этой ячейки появится виджет для настройки параметров модели с кластерным графом

In [ ]:
layout = widgets.Layout(width='500px')
style = {'description_width': '200px'}

out = widgets.Output()
status = widgets.HTML(value="<b>Готово</b>")


def run_graph_simulation(_):
    status.value = "<b style='color:orange'>Идёт расчёт...</b>"

    steps = steps_w.value
    n = n_w.value
    k = k_w.value
    p_in = p_in_w.value
    p_out = p_out_w.value
    beta = beta_w.value
    T = T_w.value

    with out:
        clear_output(wait=True)
        print("🚀 Запуск модели...")
        print("⚙️ Создание графа...")

    gen = Generator(mean=0.5, std=0.1)
    nodes = cluster_graph(gen, n, k, p_in=p_in, p_out=p_out)

    center = n // 2
    nodes[center].state = State.SPREADER

    model = Model(nodes, beta=beta, T=T)
    model.run(steps=steps, record=True)

    with out:
        clear_output(wait=True)
        display(model.plot(steps, "graph", interval=50))

    status.value = "<b style='color:green'>Готово</b>"

steps_w = widgets.IntSlider(
    value=500, min=10, max=2000, step=10,
    description="Время моделирования",
    style=style, layout=layout
)

n_w = widgets.IntSlider(
    value=50, min=10, max=200, step=5,
    description="Число вершин",
    style=style, layout=layout
)

k_w = widgets.IntSlider(
    value=4, min=1, max=20, step=1,
    description="Число кластеров",
    style=style, layout=layout
)

p_in_w = widgets.FloatSlider(
    value=0.9, min=0.0, max=1.0, step=0.01,
    description="Связь внутри кластера",
    style=style, layout=layout
)

p_out_w = widgets.FloatSlider(
    value=0.3, min=0.0, max=1.0, step=0.01,
    description="Связь между кластерами",
    style=style, layout=layout
)

beta_w = widgets.FloatSlider(
    value=0.8, min=0.0, max=1.0, step=0.01,
    description="Интересность",
    style=style, layout=layout
)

T_w = widgets.IntSlider(
    value=10, min=1, max=200, step=1,
    description="Время актуальности",
    style=style, layout=layout
)

steps_box = widgets.HBox([
    steps_w,
    widgets.Label("ч", layout=widgets.Layout(width="25px"))
])

T_box = widgets.HBox([
    T_w,
    widgets.Label("ч", layout=widgets.Layout(width="25px"))
])

button = widgets.Button(
    description="Запустить",
    button_style="success",
    icon="play",
    layout=widgets.Layout(width='200px')
)

button.on_click(run_graph_simulation)

ui = widgets.VBox([
    widgets.HBox([steps_box]),
    widgets.HBox([n_w, k_w], layout=widgets.Layout(justify_content='space-between')),
    widgets.HBox([p_in_w, p_out_w], layout=widgets.Layout(justify_content='space-between')),
    widgets.HBox([beta_w, T_box]),
    widgets.HBox([button, status]),
])

display(ui, out)

Output()